# CO-Sign Language â€” supplied training notebook
Publication edition: outputs removed for privacy; local paths replaced with COSIGN_TRAIN, COSIGN_VALIDATION and COSIGN_RESULTS. Historical training is not rerun here.
**Changed for future training:** horizontal flipping is disabled because it can swap left/right semantics without relabeling. Existing historical curves were generated before this correction.
Validation selects checkpoints; it is not an independent test set. The fixed-token ViT below has no adaptive token halting. The source dataset crop geometry is unrecorded; a newly reconstructed 350Ã—150 collector is not proof of historical crop equivalence.
Future training and preview seeds are fixed to 2025 and recorded in new checkpoints; GPU bitwise determinism is not promised. No historical metrics are changed by these edits.


# CO-Sign: PyTorch GPU training

Select **Python (ml)** and **restart the kernel**, then run the three code cells in order. Your tested PyTorch CUDA build is retained. No WSL needed.

If imports are missing, run in the activated `ml` terminal: `python -m pip install numpy scipy pillow matplotlib scikit-learn tqdm`. TensorFlow is already installed and is retained **only for CPU image augmentation/loading** to preserve the exact original transform implementation. Do not reinstall or upgrade your working PyTorch build.

## What is preserved
224Ã—224 grayscale, 7Ã—7 patches (1,024 tokens), width 64, 8 heads with **64 dimensions per head**, 12 blocks, MLP [128,64], exact GELU, dropout 0.1, global average pooling, six outputs, LR 0.001, batch 16, 10 epochs, original training augmentation.

## Explicit implementation differences
- CrossEntropyLoss receives logits; softmax is used for predictions. Same classification objective.
- Internal micro-batches of 2 accumulate into the original batch of 16 to reduce GPU memory. One optimizer update per full batch; the final partial batch is weighted correctly. This model has no batch normalization. Random masks and floating-point summation differ across frameworks, so identical learning curves are not promised. FP32 is retained; no automatic mixed precision.
- Custom attention preserves Keras per-head dimensions and initializer distributions. Stock PyTorch MultiheadAttention(64,8) would silently change the architecture.
- Adam uses eps=1e-7 and zero weight decay: your current Keras runtime ignores the obsolete `decay` argument, as shown in your screenshot. AdamW is not substituted. Framework Adam numerical details can differ.
- Validation order is fixed for plotting. It has no augmentation. Precision/recall match Keras' global 0.5-threshold definition, not macro argmax metrics.
- Best and last PyTorch checkpoints, class mapping and history are saved each completed epoch. The best validation-loss weights are restored at the end. Saving is `.pt`, not `.keras`; existing TensorFlow inference code will need a PyTorch loader.

**Publication correction:** horizontal flipping is disabled because the original enabled setting can change left/right labels. The saved historical training curves still describe the earlier run. Strong shifts/zoom should also be evaluated before new training.

**Evaluation limit:** your `my_test` folder drives validation/callbacks, so its score is not an untouched test estimate.

The model is substantial for 8 GB VRAM. Cell 1 checks actual model forward/backward on GPU before training; if it runs out of memory, restart with micro_batch_size=1. No performance guarantee is made.

References: [Keras attention dimensions](https://keras.io/api/layers/attention_layers/multi_head_attention/), [PyTorch scaled dot-product attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html).


In [ ]:
# CELL 1 â€” setup, original image loading, and the PyTorch model
# Select your existing Python (ml) kernel. Restart it before running this notebook.
# No changes to your working PyTorch/CUDA installation are needed.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import math
import json
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from tqdm.auto import tqdm

# TensorFlow/Keras is used ONLY for the original CPU image preprocessing.
# The neural network, backpropagation and optimizer below are all PyTorch.
import tensorflow as tf
try:
    tf.config.set_visible_devices([], "GPU")
except RuntimeError as exc:
    raise RuntimeError("Restart the notebook kernel, then run this cell first.") from exc
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

seed = 2025
np.random.seed(seed)
torch.manual_seed(seed)
tf.random.set_seed(seed)

TRAIN_PATH = Path(os.environ.get('COSIGN_TRAIN', 'data/train'))
TEST_PATH = Path(os.environ.get('COSIGN_VALIDATION', 'data/validation'))
OUTPUT_ROOT = Path(os.environ.get('COSIGN_RESULTS', 'runs'))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

image_size = 224
patch_size = 7
projection_dim = 64
num_heads = 8
transformer_layers = 12
mlp_head_units = [128, 64]
num_classes = 6
learning_rate = 0.001
weight_decay = 0.0001  # Retained for reference; see optimizer note in Cell 2.
num_epochs = 10
batch_size = 16
micro_batch_size = 2  # 8 micro-batches per full batch; still one update per 16 images.
dropout_rate = 0.1

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot see CUDA. Select the ml kernel that passed your GPU test.")
device = torch.device("cuda:0")
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("Training device:", torch.cuda.get_device_name(device))

for path in (TRAIN_PATH, TEST_PATH):
    if not path.is_dir():
        raise FileNotFoundError(path)
class_names = sorted(p.name for p in TRAIN_PATH.iterdir() if p.is_dir())
test_classes = sorted(p.name for p in TEST_PATH.iterdir() if p.is_dir())
if len(class_names) != num_classes or test_classes != class_names:
    raise ValueError(f"Expected the same six class folders. Train={class_names}; test={test_classes}")

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255, rotation_range=5,
    width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2,
    horizontal_flip=False, fill_mode='nearest'
)
validation_datagen = ImageDataGenerator(rescale=1.0 / 255)
train_generator = train_datagen.flow_from_directory(
    str(TRAIN_PATH), target_size=(image_size, image_size),
    batch_size=batch_size, color_mode='grayscale',
    class_mode='categorical', classes=class_names, shuffle=True, seed=seed, interpolation='nearest'
)
validation_generator = validation_datagen.flow_from_directory(
    str(TEST_PATH), target_size=(image_size, image_size),
    batch_size=batch_size, color_mode='grayscale',
    class_mode='categorical', classes=class_names, shuffle=False, seed=seed, interpolation='nearest'
)
if min(train_generator.samples, validation_generator.samples) == 0:
    raise ValueError("One dataset has no readable image filenames.")
print("Class mapping:", train_generator.class_indices)
print("Publication correction: horizontal flipping disabled to preserve left/right labels.")

class Patches(nn.Module):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def forward(self, images):
        # NCHW -> [batch, row, col, patch_row, patch_col, channel].
        # This matches tf.image.extract_patches pixel order, including channels.
        p = self.patch_size
        tiles = images.unfold(2, p, p).unfold(3, p, p)
        return tiles.permute(0, 2, 3, 4, 5, 1).contiguous().reshape(
            images.shape[0], -1, p * p * images.shape[1]
        )

class PatchEncoder(nn.Module):
    def __init__(self, num_patches, projection_dim, patch_dim):
        super().__init__()
        self.projection = nn.Linear(patch_dim, projection_dim)
        self.position_embedding = nn.Embedding(num_patches, projection_dim)

    def forward(self, patches):
        positions = torch.arange(patches.shape[1], device=patches.device)
        return self.projection(patches) + self.position_embedding(positions)

class KerasStyleAttention(nn.Module):
    def __init__(self, width, heads, key_dim):
        super().__init__()
        self.heads, self.key_dim = heads, key_dim
        # Original Keras key_dim=64 means 64 PER head: total inner width=512.
        self.query = nn.Linear(width, heads * key_dim)
        self.key = nn.Linear(width, heads * key_dim)
        self.value = nn.Linear(width, heads * key_dim)
        self.output = nn.Linear(heads * key_dim, width)

    def forward(self, x):
        b, n, _ = x.shape
        def project(layer):
            return layer(x).reshape(b, n, self.heads, self.key_dim).transpose(1, 2)
        attended = F.scaled_dot_product_attention(
            project(self.query), project(self.key), project(self.value),
            dropout_p=0.0, is_causal=False
        )
        return self.output(attended.transpose(1, 2).contiguous().reshape(b, n, -1))

    def reset_keras_parameters(self):
        # Match Glorot's fan calculation for Keras 3-D attention kernels.
        width, heads, dim = self.query.in_features, self.heads, self.key_dim
        qkv_bound = math.sqrt(6.0 / (width * heads + width * dim))
        out_bound = math.sqrt(6.0 / (heads * dim + heads * width))
        for layer in (self.query, self.key, self.value):
            nn.init.uniform_(layer.weight, -qkv_bound, qkv_bound)
            nn.init.zeros_(layer.bias)
        nn.init.uniform_(self.output.weight, -out_bound, out_bound)
        nn.init.zeros_(self.output.bias)

def mlp(in_features, hidden_units, dropout):
    layers = []
    for units in hidden_units:
        layers.extend([nn.Linear(in_features, units), nn.GELU(approximate='none'), nn.Dropout(dropout)])
        in_features = units
    return nn.Sequential(*layers)

class TransformerBlock(nn.Module):
    def __init__(self, width, heads, hidden_units, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(width, eps=1e-6)
        self.attention = KerasStyleAttention(width, heads, key_dim=width)
        self.norm2 = nn.LayerNorm(width, eps=1e-6)
        self.mlp = mlp(width, hidden_units, dropout)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class EyeViT(nn.Module):
    def __init__(self, image_size=224, patch_size=7, projection_dim=64,
                 num_heads=8, transformer_layers=12, mlp_head_units=(128, 64),
                 num_classes=6, dropout_rate=0.1):
        super().__init__()
        if mlp_head_units[-1] != projection_dim:
            raise ValueError("Residual MLP must end at projection_dim.")
        self.patches = Patches(patch_size)
        self.encoder = PatchEncoder((image_size // patch_size)**2, projection_dim, patch_size**2)
        self.blocks = nn.Sequential(*[
            TransformerBlock(projection_dim, num_heads, mlp_head_units, dropout_rate)
            for _ in range(transformer_layers)
        ])
        self.norm = nn.LayerNorm(projection_dim, eps=1e-6)
        self.head = mlp(projection_dim, mlp_head_units, dropout_rate)
        self.classifier = nn.Linear(mlp_head_units[-1], num_classes)
        self.apply(self._initialize)
        for module in self.modules():
            if isinstance(module, KerasStyleAttention):
                module.reset_keras_parameters()

    @staticmethod
    def _initialize(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.uniform_(module.weight, -0.05, 0.05)

    def forward(self, images):
        x = self.encoder(self.patches(images))
        x = self.norm(self.blocks(x)).mean(dim=1)
        # CrossEntropyLoss includes log-softmax; apply softmax only for predictions.
        return self.classifier(self.head(x))

model_config = dict(
    image_size=image_size, patch_size=patch_size, projection_dim=projection_dim,
    num_heads=num_heads, transformer_layers=transformer_layers,
    mlp_head_units=mlp_head_units, num_classes=num_classes, dropout_rate=dropout_rate
)
vit_model = EyeViT(**model_config).to(device)
print(f"Parameters: {sum(p.numel() for p in vit_model.parameters()):,}")
# Verify dimensions and patch ordering before the full-model GPU test.
assert sum(p.numel() for p in vit_model.parameters()) == 1_879_878
sample_np = np.arange(1 * 14 * 14 * 2, dtype=np.float32).reshape(1, 14, 14, 2)
tf_patches = tf.image.extract_patches(sample_np, sizes=[1, 7, 7, 1],
    strides=[1, 7, 7, 1], rates=[1, 1, 1, 1], padding='VALID').numpy().reshape(1, 4, 98)
torch_patches = Patches(7)(torch.from_numpy(sample_np).permute(0, 3, 1, 2)).numpy()
np.testing.assert_array_equal(tf_patches, torch_patches)

# Compare GPU SDPA to the explicit attention equation on a short sequence.
with torch.no_grad():
    attention = vit_model.blocks[0].attention
    sample = torch.randn(2, 17, 64, device=device)
    q, k, v = [layer(sample).reshape(2, 17, 8, 64).transpose(1, 2)
               for layer in (attention.query, attention.key, attention.value)]
    manual = ((q @ k.transpose(-2, -1)) / math.sqrt(64)).softmax(-1) @ v
    manual = attention.output(manual.transpose(1, 2).contiguous().reshape(2, 17, 512))
    torch.testing.assert_close(attention(sample), manual, rtol=1e-4, atol=2e-5)
del sample, q, k, v, manual
print("Parameter count, patch order and attention equation checks passed.")

print("Checking forward AND backward on GPU with the actual architecture...", flush=True)
vit_model.train()
try:
    probe = torch.zeros(micro_batch_size, 1, image_size, image_size, device=device)
    F.cross_entropy(vit_model(probe), torch.zeros(micro_batch_size, dtype=torch.long, device=device)).backward()
    torch.cuda.synchronize()
finally:
    vit_model.zero_grad(set_to_none=True)
print("Actual model GPU test succeeded.")
del probe
torch.cuda.empty_cache()


In [ ]:
from pathlib import Path
from tempfile import TemporaryFile

OUTPUT_ROOT = Path(os.environ.get('COSIGN_RESULTS', 'runs'))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

with TemporaryFile(dir=OUTPUT_ROOT) as test:
    test.write(b"Permission check")
    test.flush()

print("Success: Python can write to the results folder.")

In [ ]:
# CELL 2 â€” train on CUDA, save best and last checkpoints
# Run once after Cell 1. Rerunning this cell starts a new optimizer on current weights.
# To restart training from scratch, rerun Cell 1 first.
run_dir = OUTPUT_ROOT / datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
run_dir.mkdir(parents=True, exist_ok=False)
print("Saving results to:", run_dir)

# Your CURRENT Keras run warns that decay=weight_decay is ignored.
# Therefore use Adam with zero weight decay, not AdamW (which changes training).
# Adam implementations have small numerical differences between frameworks.
optimizer = torch.optim.Adam(vit_model.parameters(), lr=learning_rate,
                             betas=(0.9, 0.999), eps=1e-7, weight_decay=0.0)
criterion = nn.CrossEntropyLoss()
history = {key: [] for key in (
    'loss', 'accuracy', 'precision', 'recall',
    'val_loss', 'val_accuracy', 'val_precision', 'val_recall', 'learning_rate'
)}

class Metrics:
    def __init__(self):
        self.total = self.correct = self.tp = self.fp = self.fn = 0
        self.loss_sum = 0.0

    def update(self, logits, labels, loss):
        with torch.no_grad():
            prob = logits.detach().softmax(dim=-1)
            truth = F.one_hot(labels, num_classes=num_classes).bool()
            positive = prob > 0.5  # Match default Keras Precision/Recall, not macro averaging.
            n = labels.numel()
            self.total += n
            self.correct += (prob.argmax(1) == labels).sum().item()
            self.tp += (positive & truth).sum().item()
            self.fp += (positive & ~truth).sum().item()
            self.fn += (~positive & truth).sum().item()
            self.loss_sum += loss.item() * n

    def result(self):
        return dict(loss=self.loss_sum / self.total, accuracy=self.correct / self.total,
                    precision=self.tp / max(self.tp + self.fp, 1),
                    recall=self.tp / max(self.tp + self.fn, 1))

def run_epoch(generator, training):
    vit_model.train(training)
    metrics = Metrics()
    progress = tqdm(range(len(generator)), desc='Training' if training else 'Validation')
    for batch_index in progress:
        images_np, targets_np = generator[batch_index]
        # Original loader yields NHWC float32 normalized to [0, 1].
        images = torch.from_numpy(images_np).permute(0, 3, 1, 2).contiguous()
        labels = torch.from_numpy(targets_np.argmax(axis=1)).long()
        full_batch_size = len(labels)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            for start in range(0, full_batch_size, micro_batch_size):
                x = images[start:start+micro_batch_size].to(device)
                y = labels[start:start+micro_batch_size].to(device)
                logits = vit_model(x)
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise RuntimeError("Non-finite loss; training stopped. Earlier checkpoints are preserved.")
                if training:
                    # Weight smaller final micro-batches correctly.
                    (loss * (len(y) / full_batch_size)).backward()
                metrics.update(logits, y, loss)
        if training:
            optimizer.step()
        if batch_index % 10 == 0 or batch_index == len(generator)-1:
            values = metrics.result()
            progress.set_postfix(loss=f"{values['loss']:.4f}", accuracy=f"{values['accuracy']:.4f}")
    return metrics.result()

def save_checkpoint(path, epoch, val_loss):
    payload = {
        'model_state_dict': {k: v.detach().cpu() for k, v in vit_model.state_dict().items()},
        'optimizer_state_dict': optimizer.state_dict(),
        'model_config': model_config, 'class_names': class_names,
        'epoch': epoch, 'val_loss': val_loss, 'history': history, 'seed': seed,
        'publication_training_changes': {'horizontal_flip': False, 'deterministic_seeds': True},
        'preprocessing': {'grayscale': True, 'rescale': 1.0/255, 'resize': [image_size, image_size]},
        'batch_size': batch_size, 'micro_batch_size': micro_batch_size
    }
    temporary = path.with_suffix('.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

best_loss = float('inf')
plateau_best = float('inf')
early_wait = plateau_wait = 0
best_epoch = None

try:
    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs} | LR={optimizer.param_groups[0]['lr']:.7g}")
        lr_used = optimizer.param_groups[0]['lr']
        train_values = run_epoch(train_generator, training=True)
        val_values = run_epoch(validation_generator, training=False)
        for key, value in train_values.items():
            history[key].append(value)
        for key, value in val_values.items():
            history['val_' + key].append(value)
        history['learning_rate'].append(lr_used)
        val_loss = val_values['loss']
        improved = val_loss < best_loss
        if improved:
            best_loss, best_epoch, early_wait = val_loss, epoch, 0
        else:
            early_wait += 1
        # Match ReduceLROnPlateau: factor=.2, patience=5, min_delta=1e-4, min_lr=1e-6.
        if val_loss < plateau_best - 1e-4:
            plateau_best, plateau_wait = val_loss, 0
        else:
            plateau_wait += 1
            if plateau_wait >= 5:
                old_lr = optimizer.param_groups[0]['lr']
                if old_lr > 1e-6:
                    new_lr = max(old_lr * 0.2, 1e-6)
                    for group in optimizer.param_groups:
                        group['lr'] = new_lr
                    print(f"Reducing LR to {new_lr:.7g}")
                    plateau_wait = 0
        print(' | '.join(f"{k}: {v:.4f}" for k, v in train_values.items()))
        print(' | '.join(f"val_{k}: {v:.4f}" for k, v in val_values.items()))
        if improved:
            save_checkpoint(run_dir / 'comeback_best.pt', epoch, val_loss)
            print("Saved new best model.")
        save_checkpoint(run_dir / 'comeback_last.pt', epoch, val_loss)
        (run_dir / 'history.json').write_text(json.dumps(history, indent=2))
        (run_dir / 'class_indices.json').write_text(json.dumps(train_generator.class_indices, indent=2))
        train_generator.on_epoch_end()
        if early_wait >= 10:
            print("Early stopping: 10 epochs without validation-loss improvement.")
            break
except torch.cuda.OutOfMemoryError:
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    print("GPU memory exhausted. Set micro_batch_size=1 in Cell 1 and restart the kernel/run.")
    print("Completed epoch checkpoints, if any, remain in:", run_dir)
    raise

# Restore the best validation-loss model, as intended by restore_best_weights=True.
best_checkpoint = torch.load(run_dir / 'comeback_best.pt', map_location='cpu', weights_only=True)
vit_model.load_state_dict(best_checkpoint['model_state_dict'])
vit_model.eval()
print(f"\nRestored best model from epoch {best_epoch}; validation loss={best_loss:.4f}")
print("Best model:", run_dir / 'comeback_best.pt')
print("This is a PyTorch checkpoint, not a .keras model.")


In [ ]:
# CELL 3 â€” independent evaluation cell; run after training, no retraining.
# Can be rerun to generate ten new random augmentations.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Curves reflect metrics gathered during training, before best-weight restoration.
epochs = np.arange(1, len(history['loss']) + 1)
for metric, title, ylabel in [('loss', 'Loss', 'Cross-entropy'), ('accuracy', 'Accuracy', 'Accuracy')]:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(epochs, history[metric], label='Training')
    ax.plot(epochs, history['val_' + metric], label='Validation')
    ax.set(title=title, xlabel='Epoch', ylabel=ylabel, xticks=epochs)
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(run_dir / f'{metric}.png', dpi=180)
    plt.show()

vit_model.eval()
true_parts, predicted_parts = [], []
eval_metrics = Metrics()
with torch.inference_mode():
    for i in tqdm(range(len(validation_generator)), desc='Confusion matrix / evaluation'):
        images_np, labels_np = validation_generator[i]
        images = torch.from_numpy(images_np).permute(0, 3, 1, 2).contiguous()
        labels = torch.from_numpy(labels_np.argmax(1)).long()
        for start in range(0, len(labels), micro_batch_size):
            x = images[start:start+micro_batch_size].to(device)
            y = labels[start:start+micro_batch_size].to(device)
            logits = vit_model(x)
            eval_metrics.update(logits, y, criterion(logits, y))
            true_parts.append(y.cpu().numpy())
            predicted_parts.append(logits.argmax(1).cpu().numpy())
true_labels = np.concatenate(true_parts)
predicted_labels = np.concatenate(predicted_parts)
print("Best restored model â€” validation folder:", eval_metrics.result())
cm = confusion_matrix(true_labels, predicted_labels, labels=np.arange(num_classes))
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap='Blues', values_format='d', xticks_rotation=45, colorbar=False
)
ax.set_title('Confusion Matrix â€” Unaugmented Validation Images')
fig.tight_layout()
fig.savefig(run_dir / 'confusion_matrix.png', dpi=180)
plt.show()
np.save(run_dir / 'confusion_matrix.npy', cm)

# Match your requested preview: fresh augmentation, no left/right reversal.
preview_datagen = ImageDataGenerator(
    rescale=1.0/255, rotation_range=5,
    width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2,
    horizontal_flip=False, fill_mode='nearest'
)
rng = np.random.default_rng(seed)
indices = rng.choice(validation_generator.samples, min(10, validation_generator.samples), replace=False)
preview_images = []
for index in indices:
    image = load_img(validation_generator.filepaths[index], color_mode='grayscale',
                     target_size=(image_size, image_size), interpolation='nearest')
    array = img_to_array(image)
    array = preview_datagen.random_transform(array, seed=int(rng.integers(0, 2**31-1)))
    preview_images.append(preview_datagen.standardize(array))
preview_images = np.stack(preview_images)
preview_prob = []
with torch.inference_mode():
    for start in range(0, len(indices), micro_batch_size):
        x = torch.from_numpy(preview_images[start:start+micro_batch_size]).permute(0, 3, 1, 2).contiguous().to(device)
        preview_prob.append(vit_model(x).softmax(1).cpu().numpy())
preview_prob = np.concatenate(preview_prob)
preview_pred = preview_prob.argmax(1)
preview_true = validation_generator.classes[indices]
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    ax.axis('off')
    if i >= len(indices):
        continue
    actual, pred = preview_true[i], preview_pred[i]
    ax.imshow(preview_images[i, :, :, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'True: {class_names[actual]}\nPredicted: {class_names[pred]}\nModel score: {preview_prob[i, pred]:.1%}',
                 color='green' if actual == pred else 'red', fontsize=10)
fig.suptitle('Ten Freshly Augmented Validation Images')
fig.tight_layout(rect=[0, 0, 1, .94])
fig.savefig(run_dir / 'augmented_preview.png', dpi=180)
plt.show()
print(f'Preview: {(preview_pred == preview_true).sum()}/{len(indices)} correct; not an overall accuracy estimate.')
print('The test folder was used as validation during training, so this is not an independent final test.')
print('Graphs saved to:', run_dir)
